# So sánh Attention Map: Không DualGatedFusion vs Có DualGatedFusion

Notebook này trực quan hóa sự khác biệt của SAN attention map khi:
- **Không dùng DGF**: Image tokens đi thẳng vào SAN
- **Có DGF**: Image tokens được điều chỉnh bởi question context trước khi vào SAN

**Các bước:**
1. Import & setup
2. Định nghĩa helper classes
3. Load dataset & chọn ảnh mẫu
4. Khởi tạo model & gắn hooks
5. Tính attention maps
6. Visualize kết quả

## Cell 1 – Import thư viện

In [ ]:
import os, sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

# Đảm bảo import từ thư mục project
sys.path.insert(0, os.path.abspath('.'))

from config import config
from features_extraction import ImageEmbedding, QuesEmbedding
from sans import StackedAttentionNets
from vqa_model import DualGatedFusion

# SigLIP patch 16×16 trên ảnh 224×224 → grid 14×14 = 196 patches
GRID_H, GRID_W = 14, 14
NUM_SAMPLES    = 3   # ← thay đổi số ảnh muốn xem
NUM_ATT_LAYERS = 4   # ← số SAN layers khớp với model của bạn

print(f'Device: {config.DEVICE}')
print('Import OK!')

## Cell 2 – Định nghĩa AttentionHook và VQAEncoder

In [ ]:
class AttentionHook:
    """Bắt attention weights từ ff_attention của StackedAttentionNets."""
    def __init__(self):
        self.weights = []

    def hook_fn(self, module, inp, out):
        # out: [B, N, 1] -> softmax -> [B, N]
        pi = F.softmax(out.squeeze(-1), dim=1)
        self.weights.append(pi.detach().cpu())

    def clear(self):
        self.weights.clear()


class VQAEncoder(nn.Module):
    """Chỉ chạy phần encoder: Image + Ques → (DGF?) → SAN"""
    def __init__(self, use_dual_gating: bool, num_att_layers: int = 4):
        super().__init__()
        self.use_dual_gating = use_dual_gating
        self.image_model       = ImageEmbedding(output_size=config.d_model).to(config.DEVICE)
        self.ques_model        = QuesEmbedding(output_size=config.d_model).to(config.DEVICE)
        self.dual_gated_fusion = DualGatedFusion(d_model=config.d_model).to(config.DEVICE)
        self.san_model = nn.ModuleList(
            [StackedAttentionNets(d=config.d_model, k=768) for _ in range(num_att_layers)]
        ).to(config.DEVICE)

    def _to_patch_tensor(self, out, batch_size):
        if torch.is_tensor(out):
            emb = out
        elif hasattr(out, 'last_hidden_state') and out.last_hidden_state is not None:
            emb = out.last_hidden_state
        else:
            emb = out.pooler_output
        return emb if emb.dim() == 3 else emb.reshape(batch_size, 768, -1).permute(0, 2, 1)

    def forward(self, images, questions):
        B = images.size(0)
        img_emb  = self._to_patch_tensor(self.image_model(images.to(config.DEVICE)), B)
        ques_emb = self.ques_model(questions).unsqueeze(1)

        if self.use_dual_gating:
            img_emb, ques_emb = self.dual_gated_fusion(
                img_emb.to(config.DEVICE), ques_emb.to(config.DEVICE))

        out = None
        for layer in self.san_model:
            out = layer(img_emb.to(config.DEVICE), ques_emb.to(config.DEVICE))
        return out


@torch.no_grad()
def get_attention_maps(model, hook, img_tensors, questions):
    """Trả về attention map trung bình qua tất cả SAN layers. Shape: [B, N]"""
    hook.clear()
    model(img_tensors, questions)
    stacked = torch.stack(hook.weights, dim=0)  # [L, B, N]
    return stacked.mean(dim=0)                  # [B, N]


print('Classes defined OK!')

## Cell 3 – Load dataset & chuẩn bị ảnh mẫu

In [ ]:
from datasets import load_dataset

print('Đang load dataset (lần đầu có thể mất vài phút)...')
raw_dataset = load_dataset(config.DATASET_NAME)
test_split  = raw_dataset['test']
print(f'Test split: {len(test_split)} samples')

image_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

indices     = list(range(min(NUM_SAMPLES, len(test_split))))
raw_imgs    = [test_split[i]['image'].convert('RGB') for i in indices]
questions   = [test_split[i]['question']              for i in indices]
answers     = [test_split[i]['answer']                for i in indices]
img_tensors = torch.stack([image_tf(img) for img in raw_imgs])  # [B,3,224,224]

# Hiển thị ảnh mẫu
fig, axes = plt.subplots(1, len(raw_imgs), figsize=(6 * len(raw_imgs), 5))
if len(raw_imgs) == 1:
    axes = [axes]
for ax, img, q, a in zip(axes, raw_imgs, questions, answers):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Q: {q}\nA: {a}', fontsize=10, wrap=True)
plt.suptitle('Ảnh mẫu từ test split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Load dữ liệu xong!')

## Cell 4 – Khởi tạo model và gắn attention hooks

In [ ]:
print('Khởi tạo 2 encoder (có thể mất ~1-2 phút lần đầu)...')

model_no_dgf  = VQAEncoder(use_dual_gating=False, num_att_layers=NUM_ATT_LAYERS).eval()
model_yes_dgf = VQAEncoder(use_dual_gating=True,  num_att_layers=NUM_ATT_LAYERS).eval()

# Đồng bộ trọng số để so sánh công bằng
model_yes_dgf.image_model.load_state_dict(model_no_dgf.image_model.state_dict())
model_yes_dgf.ques_model.load_state_dict(model_no_dgf.ques_model.state_dict())
for l_no, l_yes in zip(model_no_dgf.san_model, model_yes_dgf.san_model):
    l_yes.load_state_dict(l_no.state_dict())

# Gắn hooks vào ff_attention của mỗi SAN layer
hook_no  = AttentionHook()
hook_yes = AttentionHook()

for layer in model_no_dgf.san_model:
    layer.ff_attention.register_forward_hook(hook_no.hook_fn)
for layer in model_yes_dgf.san_model:
    layer.ff_attention.register_forward_hook(hook_yes.hook_fn)

print('Model sẵn sàng!')
total_no  = sum(p.numel() for p in model_no_dgf.parameters())
total_yes = sum(p.numel() for p in model_yes_dgf.parameters())
print(f'  Params (no  DGF): {total_no:,}')
print(f'  Params (yes DGF): {total_yes:,}')

## Cell 5 – Tính attention maps

In [ ]:
print('Đang tính attention maps...')

attn_no  = get_attention_maps(model_no_dgf,  hook_no,  img_tensors, questions)
attn_yes = get_attention_maps(model_yes_dgf, hook_yes, img_tensors, questions)

print(f'Shape attention (no  DGF): {attn_no.shape}   # [B, {GRID_H*GRID_W}]')
print(f'Shape attention (yes DGF): {attn_yes.shape}')
print('Tính xong!')

## Cell 6 – Helper: overlay heatmap lên ảnh

In [ ]:
def overlay_heatmap(img_np, attn_1d, alpha=0.55, cmap_name='jet'):
    """
    img_np  : np.ndarray [H, W, 3] float32 [0,1]
    attn_1d : np.ndarray [N]  (N = 196 patches)
    Returns : blended [H, W, 3], attn_resized [H, W]
    """
    attn_map = attn_1d.reshape(GRID_H, GRID_W)
    attn_map = (attn_map - attn_map.min()) / (attn_map.max() - attn_map.min() + 1e-8)

    H, W = img_np.shape[:2]
    t = torch.tensor(attn_map).unsqueeze(0).unsqueeze(0)       # [1,1,14,14]
    t = F.interpolate(t, size=(H, W), mode='bilinear', align_corners=False)
    attn_resized = t.squeeze().numpy()                          # [H, W]

    heatmap = cm.get_cmap(cmap_name)(attn_resized)[:, :, :3]  # [H, W, 3]
    blended = np.clip((1 - alpha) * img_np + alpha * heatmap, 0, 1)
    return blended, attn_resized

print('Helper functions OK!')

## Cell 7 – Visualize: So sánh từng ảnh

In [ ]:
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

for i, (img_pil, q, a) in enumerate(zip(raw_imgs, questions, answers)):
    img_np = np.array(img_pil.resize((224, 224))).astype(np.float32) / 255.0

    attn_no_np  = attn_no[i].numpy()
    attn_yes_np = attn_yes[i].numpy()

    overlay_no,  map_no  = overlay_heatmap(img_np, attn_no_np)
    overlay_yes, map_yes = overlay_heatmap(img_np, attn_yes_np)
    diff = map_yes - map_no  # dương = DGF chú ý thêm; âm = bớt

    fig, axes = plt.subplots(1, 4, figsize=(22, 5))

    # Cột 0: Ảnh gốc
    axes[0].imshow(img_np)
    axes[0].set_title('Ảnh gốc', fontsize=12, fontweight='bold')
    axes[0].set_xlabel(f'Q: "{q}"\nA: "{a}"', fontsize=9)
    axes[0].axis('off')

    # Cột 1: Không DGF
    axes[1].imshow(overlay_no)
    axes[1].set_title('Attention – Không DGF', fontsize=12, fontweight='bold', color='#c0392b')
    axes[1].axis('off')

    # Cột 2: Có DGF
    axes[2].imshow(overlay_yes)
    axes[2].set_title('Attention – Có DGF', fontsize=12, fontweight='bold', color='#27ae60')
    axes[2].axis('off')

    # Cột 3: Diff map
    vmax = max(abs(diff.max()), abs(diff.min()), 1e-6)
    im = axes[3].imshow(diff, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[3].set_title('Diff (DGF − Không DGF)', fontsize=12, fontweight='bold')
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

    plt.suptitle(
        f'Mẫu {i+1}/{len(raw_imgs)}  |  '
        'Đỏ/vàng = chú ý nhiều  |  '
        'Diff: xanh = DGF chú ý thêm, đỏ = DGF bớt chú ý',
        fontsize=10, y=1.01
    )
    plt.tight_layout()
    plt.show()

## Cell 8 – (Tùy chọn) Xem attention map thuần – không blend với ảnh

In [ ]:
# Visualize raw attention map dưới dạng grid 14×14 cho từng SAN layer
print('Re-running để lấy attention theo từng layer...')

hook_no.clear()
hook_yes.clear()

with torch.no_grad():
    model_no_dgf(img_tensors, questions)
    model_yes_dgf(img_tensors, questions)

num_layers = len(hook_no.weights)
sample_idx = 0  # ← đổi index mẫu muốn xem
img_np = np.array(raw_imgs[sample_idx].resize((224, 224))).astype(np.float32) / 255.0

fig, axes = plt.subplots(2, num_layers, figsize=(5 * num_layers, 9))
row_labels = ['Không DGF', 'Có DGF']
hooks_list = [hook_no, hook_yes]

for row, (label, hook) in enumerate(zip(row_labels, hooks_list)):
    for col, layer_attn in enumerate(hook.weights):
        # layer_attn: [B, N]
        a = layer_attn[sample_idx].numpy().reshape(GRID_H, GRID_W)
        a = (a - a.min()) / (a.max() - a.min() + 1e-8)
        axes[row, col].imshow(a, cmap='hot', vmin=0, vmax=1)
        axes[row, col].set_title(f'Layer {col+1}', fontsize=11)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(label, fontsize=13, fontweight='bold', rotation=90, labelpad=10)

plt.suptitle(
    f'Raw Attention Grid (14×14) theo từng SAN Layer – Mẫu {sample_idx+1}\n'
    f'Q: "{questions[sample_idx]}"',
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.show()